# Training model Self-Checkout

Notebook ini menyusun dataset dari foto kita, lalu melatih (fine-tuning) model YOLO nano. Hasilnya file `best.pt`.
Fine-tuning artinya model yang sudah pintar mengenali benda umum dilatih lagi dengan foto kita, supaya mengenali 8 produk kita.

Langkah singkat:
1. Di komputer, kompres tiga folder dari `data/` menjadi satu file `foto.zip`: `product_photos`, `scene_photos_train`, dan `background_photos`. Jangan ikutkan `scene_photos_test`, karena foto tes tidak boleh dipakai belajar.
2. Upload `foto.zip` ke Google Drive, ke folder `MyDrive/self-checkout/`.
3. Di Colab, pilih Runtime, lalu Change runtime type, lalu pilih GPU. Tanpa GPU training sangat lambat.
4. Jalankan sel di bawah satu per satu dari atas.

## 1. Pengaturan

`IMG_SIZE` harus sama dengan `IMG_SIZE` di `config.py`, karena model harus melihat ukuran gambar yang sama saat training dan saat dipakai di aplikasi.
`CLASS_NAMES` harus sama dengan urutan baris di `products.csv`, karena urutan itu menjadi nomor kelas di file label.

In [ ]:
MODEL = "yolo26n.pt"
IMG_SIZE = 640
EPOCHS = 100
PATIENCE = 20
BATCH = 16
ZIP_PATH = "/content/drive/MyDrive/self-checkout/foto.zip"
OUTPUT_DIR = "/content/drive/MyDrive/self-checkout"
CLASS_NAMES = [
    "aqua_600ml",
    "le_minerale_600ml",
    "walens_blueberry",
    "cussons_baby_wipes",
    "hansaplast_10",
    "kahf_revered",
    "soffell_kulit_jeruk",
    "vaseline_uv_brightening",
]

## 2. Cek GPU dan pasang library

Hasil `GPU: True` artinya GPU aktif. Kalau `False`, ganti runtime dulu.

In [ ]:
import torch
print("GPU:", torch.cuda.is_available())

In [ ]:
!pip install -q "ultralytics>=8.4,<8.5"

## 3. Ambil foto dari Google Drive

Sel pertama meminta izin akses Google Drive. Sel kedua mengekstrak zip.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!rm -rf /content/foto
!mkdir -p /content/foto
!unzip -q "{ZIP_PATH}" -d /content/foto

## 4. Susun dataset

YOLO membaca foto dari folder `images/train` dan `images/val`, dan label dari `labels/train` dan `labels/val`. Sel ini menyalin foto ke susunan itu dan membaginya jadi train (untuk belajar) dan val (untuk memantau saat training).

Pembagiannya per sesi atau per adegan, tidak acak per foto. Foto burst hampir kembar, jadi kalau dibagi acak, foto yang hampir sama ada di train dan val sekaligus dan hasil val tampak bagus padahal model hanya menghafal.

1. Foto produk: tiap produk punya dua sesi. Sesi dengan foto paling sedikit jadi val, sisanya train.
2. Adegan: setiap adegan kelima jadi val.
3. Foto latar (HP, dompet, tangan, meja kosong): sesi dengan foto paling sedikit jadi val. Label foto latar dikosongkan, artinya "tidak ada produk".
4. Hanya foto yang punya file label yang dipakai. Foto adegan tanpa label dilewati.

In [ ]:
from pathlib import Path
import shutil

# Isi zip kadang berada di dalam satu folder induk, jadi cari folder product_photos di mana pun.
found = next(Path("/content/foto").rglob("product_photos"), None)
assert found is not None, "Folder product_photos tidak ditemukan. Cek isi foto.zip."
RAW = found.parent
print("Foto dibaca dari", RAW)
OUT = Path("/content/dataset")
shutil.rmtree(OUT, ignore_errors=True)


def has_label(photo):
    label = photo.with_suffix(".txt")
    return label.exists() and label.read_text().strip() != ""


def split_by_session(photos):
    """Sesi dengan foto paling sedikit jadi val. Kalau cuma satu sesi, semuanya train."""
    sessions = {}
    for photo in photos:
        sessions.setdefault("_".join(photo.stem.split("_")[:2]), []).append(photo)
    if len(sessions) < 2:
        return photos, []
    val = sessions.pop(min(sessions, key=lambda s: len(sessions[s])))
    return [p for group in sessions.values() for p in group], val


def copy_photos(photos, prefix, split, background=False):
    for photo in photos:
        name = f"{prefix}__{photo.name}"
        (OUT / "images" / split).mkdir(parents=True, exist_ok=True)
        (OUT / "labels" / split).mkdir(parents=True, exist_ok=True)
        shutil.copy(photo, OUT / "images" / split / name)
        label = OUT / "labels" / split / (Path(name).stem + ".txt")
        if background:
            label.write_text("")
        else:
            shutil.copy(photo.with_suffix(".txt"), label)


for class_name in CLASS_NAMES:
    photos = [p for p in sorted((RAW / "product_photos" / class_name).glob("*.jpg")) if has_label(p)]
    train, val = split_by_session(photos)
    copy_photos(train, class_name, "train")
    copy_photos(val, class_name, "val")

scenes = [p for p in sorted((RAW / "scene_photos_train").glob("*.jpg")) if has_label(p)]
copy_photos([p for i, p in enumerate(scenes) if i % 5 != 4], "scene", "train")
copy_photos([p for i, p in enumerate(scenes) if i % 5 == 4], "scene", "val")

train, val = split_by_session(sorted((RAW / "background_photos").glob("*_bg*.jpg")))
copy_photos(train, "bg", "train", background=True)
copy_photos(val, "bg", "val", background=True)

names = "\n".join(f"  {i}: {name}" for i, name in enumerate(CLASS_NAMES))
yaml_path = OUT / "data.yaml"
yaml_path.write_text(f"path: {OUT}\ntrain: images/train\nval: images/val\nnames:\n{names}\n")
for split in ("train", "val"):
    print(split, len(list((OUT / "images" / split).glob("*.jpg"))), "foto")

## 5. Training

Training berhenti sendiri kalau hasil val tidak membaik selama `PATIENCE` epoch, jadi tidak selalu sampai `EPOCHS`.

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL)
model.train(
    data=str(yaml_path),
    imgsz=IMG_SIZE,
    epochs=EPOCHS,
    patience=PATIENCE,
    batch=BATCH,
    project="/content/runs",
    name="self_checkout",
    exist_ok=True,
)

## 6. Hasil

Angka di sini dihitung dari foto val, yaitu sesi dan adegan yang tidak dipakai untuk belajar. Catat angkanya apa adanya.

Angka yang paling jujur untuk laporan tetap berasal dari foto adegan tes, yang tidak pernah ikut training.

In [ ]:
best = YOLO("/content/runs/self_checkout/weights/best.pt")
metrics = best.val(data=str(yaml_path), imgsz=IMG_SIZE)
print("mAP50 semua kelas:", round(float(metrics.box.map50), 3))
print("mAP50-95 semua kelas:", round(float(metrics.box.map), 3))
print()
for index, class_index in enumerate(metrics.box.ap_class_index):
    print(best.names[int(class_index)], "mAP50:", round(float(metrics.box.ap50[index]), 3))

In [ ]:
from IPython.display import Image, display
display(Image(filename="/content/runs/self_checkout/results.png"))
display(Image(filename="/content/runs/self_checkout/confusion_matrix.png"))

## 7. Simpan hasil

`best.pt` disalin ke Google Drive dan juga diunduh ke komputer. Taruh file itu di folder utama repo, di samping `main.py`.

In [ ]:
!mkdir -p "{OUTPUT_DIR}"
!cp /content/runs/self_checkout/weights/best.pt "{OUTPUT_DIR}/best.pt"
!cp /content/runs/self_checkout/results.png "{OUTPUT_DIR}/results.png"
!cp /content/runs/self_checkout/confusion_matrix.png "{OUTPUT_DIR}/confusion_matrix.png"

from google.colab import files
files.download("/content/runs/self_checkout/weights/best.pt")

## 8. Setelah ini

1. Taruh `best.pt` di folder utama repo, di samping `main.py`.
2. Jalankan aplikasi dengan `python main.py`.